# 第 4 周练习 —— Python 转 JavaScript 代码转换器

基于 **Gradio** 的小工具：用前沿大模型把 Python 转成面向性能的 JavaScript，并可并排执行对比。

## 练习目标（理念）

- 用 AI 把 Python 转成 JavaScript
- 并排执行 Python / JavaScript，核对输出是否一致
- 经 **OpenRouter**（或直连 OpenAI）切换多种模型
- 顺带观察不同实现的性能差异

## 为什么选 JavaScript？

- 运行时通用（浏览器 + Node.js）
- 无需单独编译步骤
- 便于测试与部署
- 对 Web 开发与 serverless 场景很实用

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user 提示词 | `SYSTEM_PROMPT` + `create_user_prompt` |
| 工具型 UI | Gradio Blocks：转换 + 运行 |
| 本地执行 | `exec` 跑 Python；`node main.js` 跑 JS |

## 怎么跑

1. 配置 `.env`：`OPENAI_API_KEY`（OpenRouter 的 `sk-or-*` 或官方密钥）
2. 本机安装 Node.js（否则无法「Run JavaScript」）
3. 从上到下运行单元格，最后 `app.launch()`
4. 在 UI 里选示例或自写 Python → Convert → 两边 Run 对比输出


In [ ]:
# ========== 导入：Python→JS 转换器要用的工具箱 ==========

# os：读环境变量，例如 OPENAI_API_KEY
import os
# io：StringIO 捕获 exec 时的标准输出
import io
# sys：临时替换 sys.stdout，把 print 重定向到缓冲区
import sys
# json：把 system_info 等字典格式化进 prompt
import json
# shutil：which("node") 探测 Node.js 是否在 PATH
import shutil
# subprocess：调用 node --version / node main.js
import subprocess
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI 客户端：也可改 base_url 指向 OpenRouter
from openai import OpenAI
# gradio：搭转换与双端执行的 Web UI
import gradio as gr


In [ ]:
# ========== 环境检查：API Key + 本机是否安装 Node.js ==========

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenAI / OpenRouter 共用的密钥名
openai_api_key = os.getenv('OPENAI_API_KEY')

# 状态打印文案保持英文原样（运行时可见）
if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# 在 PATH 里找 node 可执行文件
node_path = shutil.which("node")
if node_path:
    try:
        # 子进程跑 node --version；text=True 直接得 str
        version = subprocess.check_output(["node", "--version"], text=True).strip()
        print(f"Node.js installed: {version}")
    except:
        # which 找到了但 --version 失败时的提示
        print("Node.js found but version check failed")
else:
    # 没装 Node：后面 JS 执行会不可用
    print("Node.js not installed - JavaScript execution will be disabled")


In [ ]:
# ========== 客户端与模型列表：OpenRouter 或直连 OpenAI ==========

# 密钥以 sk-or- 开头则走 OpenRouter；否则默认官方 OpenAI
if openai_api_key and openai_api_key.startswith('sk-or-'):
    # base_url 指向 OpenRouter 的 OpenAI 兼容端点（URL 禁止改译）
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("Using OpenRouter")
else:
    # 无前缀或非 OpenRouter：用默认基址 + 环境变量密钥
    client = OpenAI()
    print("Using OpenAI directly")

# 下拉可用的 model id（经 OpenRouter 时这些名字才有意义；禁止改译）
MODELS = [
    "gpt-4.1-mini",
    "gpt-4.1",
    "anthropic/claude-sonnet-4",
    "google/gemini-2.0-flash-exp:free",
    "deepseek/deepseek-coder",
    "qwen/qwen-2.5-coder-32b-instruct",
]

# 打印可选模型个数，确认列表已加载
print(f"Available models: {len(MODELS)}")


In [ ]:
# ========== 收集本机信息：塞进 user prompt，帮助模型生成可运行的 JS ==========

# platform：查 OS / 架构 / Python 版本等
import platform

def get_system_info():
    """Gather system information for the LLM context."""
    # 用字典汇总，稍后 json.dumps 进提示词
    info = {
        "os": platform.system(),
        "os_version": platform.version(),
        "architecture": platform.machine(),
        "python_version": platform.python_version(),
        # which("node") 非空即认为已安装
        "node_installed": bool(shutil.which("node")),
    }
    
    # 若有 Node，再尽量读出版本号
    if info["node_installed"]:
        try:
            info["node_version"] = subprocess.check_output(
                ["node", "--version"], text=True
            ).strip()
        except:
            # 版本命令失败就标 unknown，不中断主流程
            info["node_version"] = "unknown"
    
    return info

# 模块级算一次，后面 create_user_prompt 直接引用
system_info = get_system_info()
# 漂亮打印 JSON，方便你在笔记本里核对本机环境
print(json.dumps(system_info, indent=2))


In [ ]:
# ========== 提示词：system 定转换规则，user 塞系统信息 + Python 源码 ==========

# SYSTEM_PROMPT：要求只输出 JS、输出与 Python 一致等（整段 prompt 禁止改译）
SYSTEM_PROMPT = """
Your task is to convert Python code into high-performance JavaScript code.
Respond only with JavaScript code. Do not provide any explanation other than occasional comments.

Requirements:
1. The JavaScript code must produce IDENTICAL output to the Python code
2. Optimize for fastest possible execution
3. Use modern JavaScript (ES6+) features
4. The code should run in Node.js
5. Use console.log() for output (equivalent to print())
6. Handle Python-specific constructs appropriately:
   - range() -> for loops or Array.from()
   - list comprehensions -> map/filter or loops
   - f-strings -> template literals
   - time.time() -> performance.now() or Date.now()

Important:
- Ensure numeric precision matches Python output
- Handle large numbers correctly (use BigInt if needed)
- Output should be exactly the same as Python
"""

def create_user_prompt(python_code: str) -> str:
    """Create the user prompt for code conversion."""
    # f-string 拼 user 内容：含 system_info、运行方式说明、以及 Python 代码块
    return f"""
Convert this Python code to JavaScript that produces identical output.

System information:
{json.dumps(system_info, indent=2)}

The JavaScript will be saved to main.js and run with: node main.js

Respond only with JavaScript code. No markdown fences or explanation.

Python code:

```python
{python_code}
```
"""


In [ ]:
# ========== 转换函数：调用所选模型，并把可能的 markdown 围栏剥掉 ==========

def convert_to_javascript(model: str, python_code: str) -> str:
    """Convert Python code to JavaScript using the selected model."""
    
    # 空输入：返回占位注释（英文原样，UI 可能直接显示）
    if not python_code.strip():
        return "// No Python code provided"
    
    try:
        # messages：system 规则 + user（含源码与本机信息）
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": create_user_prompt(python_code)}
        ]
        
        # Chat Completions；低温减少随意改写；max_tokens 给足生成空间
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=4096,
            temperature=0.2
        )
        
        # content 可能为 None，用空串兜底
        js_code = response.choices[0].message.content or ""
        
        # 模型常包 ```javascript / ```js / ```；简单 replace 去掉
        for fence in ["```javascript", "```js", "```"]:
            js_code = js_code.replace(fence, "")
        
        # 去掉首尾空白后返回给 Gradio 代码框
        return js_code.strip()
        
    except Exception as e:
        # 失败时仍返回可显示的 JS 注释行，避免 UI 崩
        return f"// Error: {str(e)}"


In [ ]:
# ========== 双端执行：exec 跑 Python；写 main.js 后用 node 跑 ==========

def run_python(code: str) -> str:
    """Execute Python code and capture output."""
    # 空代码：返回提示文案（原样保留）
    if not code.strip():
        return "No code to execute"
    
    # 受限全局命名空间：只暴露 builtins，减少意外依赖
    globals_dict = {"__builtins__": __builtins__}
    # 内存缓冲区：承接被重定向的 stdout
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    
    try:
        # 动态执行用户/示例 Python；print 会进 buffer
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        # 执行错误也收成字符串给 UI
        output = f"Error: {e}"
    finally:
        # 无论成败都恢复真正的 stdout
        sys.stdout = old_stdout
    
    # 没打印任何东西时给占位，避免空白框难辨
    return output if output else "(no output)"


def run_javascript(code: str) -> str:
    """Execute JavaScript code using Node.js."""
    if not code.strip():
        return "No code to execute"
    
    # 没有 node：明确告诉用户去哪安装（URL 原样）
    if not shutil.which("node"):
        return "Error: Node.js is not installed. Install it from https://nodejs.org/"
    
    # 把 JS 落到磁盘 main.js，供 node 读取
    try:
        with open("main.js", "w") as f:
            f.write(code)
        
        # 子进程执行；capture_output 收 stdout/stderr；超时 30 秒
        result = subprocess.run(
            ["node", "main.js"],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        # 非零退出码：把 stderr 展示为错误
        if result.returncode != 0:
            return f"Error:\n{result.stderr}"
        
        return result.stdout if result.stdout else "(no output)"
        
    except subprocess.TimeoutExpired:
        # 超时文案原样保留
        return "Error: Execution timed out (30s limit)"
    except Exception as e:
        return f"Error: {str(e)}"


In [ ]:
# ========== 示例算法：π / 斐波那契 / 素数筛（字符串内容禁止改动） ==========

# Leibniz 级数近似 π：作为默认示例喂给转换器
EXAMPLE_PI = '''
import time

def calculate_pi(iterations):
    """Calculate pi using Leibniz formula."""
    result = 0.0
    for i in range(iterations):
        sign = 1 if i % 2 == 0 else -1
        result += sign / (2 * i + 1)
    return result * 4

start = time.time()
pi = calculate_pi(1_000_000)
end = time.time()

print(f"Pi approximation: {pi:.10f}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

# 迭代法斐波那契：对比语言在循环上的表现
EXAMPLE_FIBONACCI = '''
import time

def fibonacci(n):
    """Calculate nth Fibonacci number iteratively."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

start = time.time()
result = fibonacci(40)
end = time.time()

print(f"Fibonacci(40) = {result}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

# 埃氏筛数素数：偏计算型，适合看 JS 优化空间
EXAMPLE_PRIMES = '''
import time

def sieve_of_eratosthenes(limit):
    """Find all primes up to limit using Sieve of Eratosthenes."""
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False
    
    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, limit + 1, i):
                is_prime[j] = False
    
    return sum(is_prime)

start = time.time()
count = sieve_of_eratosthenes(1_000_000)
end = time.time()

print(f"Primes up to 1,000,000: {count}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

# 下拉展示名 → 示例源码（展示名字符串保持英文原样）
EXAMPLES = {
    "Pi Calculation (Leibniz)": EXAMPLE_PI,
    "Fibonacci (Iterative)": EXAMPLE_FIBONACCI,
    "Prime Sieve": EXAMPLE_PRIMES
}

# 确认三个示例键名已注册
print(f"Example algorithms: {list(EXAMPLES.keys())}")


In [ ]:
# ========== Gradio UI：选示例/模型 → 转换 → 双端运行对比 ==========

def load_example(example_name: str) -> str:
    """Load an example Python code snippet."""
    # 下拉变更时：按名字取示例；没有则空串
    return EXAMPLES.get(example_name, "")


# Soft 主题；title 为浏览器标签（UI 字符串原样）
with gr.Blocks(title="Python to JavaScript Converter", theme=gr.themes.Soft()) as app:
    # 页头说明（Markdown 字符串是 UI 内容，保持英文原样）
    gr.Markdown("""
    # Python to JavaScript Code Converter
    
    Convert Python code to optimized JavaScript using AI models.
    Execute both versions to compare output and performance.
    """)
    
    # 第一行：示例下拉 + 模型下拉
    with gr.Row():
        example_dropdown = gr.Dropdown(
            choices=list(EXAMPLES.keys()),
            label="Load Example",
            value=list(EXAMPLES.keys())[0]
        )
        model_dropdown = gr.Dropdown(
            choices=MODELS,
            value=MODELS[0],
            label="Model"
        )
    
    # 第二行：左 Python 编辑器 / 右 JS 编辑器
    with gr.Row(equal_height=True):
        with gr.Column():
            python_code = gr.Code(
                label="Python Code",
                language="python",
                lines=20,
                value=EXAMPLE_PI.strip()
            )
        with gr.Column():
            js_code = gr.Code(
                label="JavaScript Code",
                language="javascript",
                lines=20,
                value="// Click 'Convert to JavaScript' to generate code"
            )
    
    # 第三行：运行 Python / 转换 / 运行 JavaScript
    with gr.Row():
        run_python_btn = gr.Button("Run Python", variant="secondary")
        convert_btn = gr.Button("Convert to JavaScript", variant="primary")
        run_js_btn = gr.Button("Run JavaScript", variant="secondary")
    
    # 第四行：两侧输出文本框（只读）
    with gr.Row(equal_height=True):
        with gr.Column():
            python_output = gr.Textbox(
                label="Python Output",
                lines=6,
                interactive=False
            )
        with gr.Column():
            js_output = gr.Textbox(
                label="JavaScript Output",
                lines=6,
                interactive=False
            )
    
    # 底部使用提示（UI 文案原样）
    gr.Markdown("""
    ### Notes
    - JavaScript execution requires Node.js to be installed
    - Output should be identical between Python and JavaScript
    - Performance may vary based on the algorithm and implementation
    """)
    
    # 事件：换示例 → 填入 Python 代码框
    example_dropdown.change(
        fn=load_example,
        inputs=[example_dropdown],
        outputs=[python_code]
    )
    
    # 事件：转换按钮 → 调用 LLM 写 JS
    convert_btn.click(
        fn=convert_to_javascript,
        inputs=[model_dropdown, python_code],
        outputs=[js_code]
    )
    
    # 事件：本地 exec 跑 Python
    run_python_btn.click(
        fn=run_python,
        inputs=[python_code],
        outputs=[python_output]
    )
    
    # 事件：node 跑 JavaScript
    run_js_btn.click(
        fn=run_javascript,
        inputs=[js_code],
        outputs=[js_output]
    )


In [ ]:
# ========== 启动 Gradio 应用 ==========

# launch：阻塞式打开本地 Web UI（参数保持原样，未擅自加减）
app.launch()


## 用法指南

### 怎么用

1. **Load Example**：选示例，或自己在左侧写 Python
2. **Select Model**：选用来做转换的 AI 模型
3. **Run Python**：先跑 Python，得到期望输出
4. **Convert to JavaScript**：生成等价 JavaScript
5. **Run JavaScript**：执行并与 Python 输出对比

### 环境要求

- Python 3.8+
- Node.js（用于执行 JavaScript）
- OpenRouter API Key（或 OpenAI API Key）

### 支持的 Python 写法（转换侧约定）

- 基础算术与逻辑
- 函数与循环
- 列表推导式（list comprehensions）
- f-string → 模板字符串（template literals）
- `time` 模块 → `performance.now()` 等

### 性能说明

- 同一算法在 JS 上可能更快或更慢，取决于实现
- Node.js 的 JIT 有时能带来明显加速
- 大整数处理在两种语言间可能不同，需留意精度
